# Cracked Surface Classification with VGG19

A TensorFlow/Keras notebook exploring a VGG19-based image classifier for cracked versus non-cracked surfaces. It includes training, checkpoint, confusion-matrix, and ROC-curve sections. The notebook does not include the image dataset or trained weights.

**Data and reproducibility:** Image files arranged in two class directories (cracked and non-cracked) are required. The notebook has placeholder checkpoint/model paths that need to be set to valid local paths before training or loading a saved model.

Install the dependencies listed in `requirements.txt` in a compatible Python environment before running the notebook. The saved outputs were cleared; training and evaluation cells have not been rerun as part of this cleanup.


# **VGG19**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten,GlobalAveragePooling2

In [ ]:
# Load the pre-trained VGG-19 model without the top classification layers

base_model = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3)) #Update Input Shape
base_model.trainable = False ## Not trainable weights
base_model.summary()

In [ ]:
flatten_layer = layers.GlobalAveragePooling2D()
dense_layer_1 = layers.Dense(128, activation='relu')
dense_layer_2 = layers.Dense(64, activation='relu')
prediction_layer = layers.Dense(4, activation='softmax')

model = models.Sequential([
    base_model,
    flatten_layer,
    prediction_layer
])

model.summary()

In [ ]:
from tensorflow.keras.optimizers import SGD, Adam

sgd = SGD(lr=0.001,decay=1e-6, momentum=0.9, nesterov=True)

# We are going to use accuracy metrics and cross entropy loss as performance parameters
from tensorflow.keras.losses import binary_crossentropy / categorical_crossentropy
model.compile(
     optimizer = Adam(learning_rate = 0.001),
     loss='categorical_crossentropy / binary_crossentropy',
     metrics=['accuracy']
    )
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Define the EarlyStopping callback
early_stopping = EarlyStopping(
    patience=15,
    min_delta=0.001,
    monitor="val_acc",
    restore_best_weights=True
)


# Define the ModelCheckpoint callback to save the best weights
checkpoint = ModelCheckpoint(
    'checkpoints/EPOCH.h5', 
    monitor='val_acc',
    save_best_only=True
)

In [ ]:
# Train the model
history = model.fit(
     X_train,
     y_train,
     epochs=200,
     batch_size=16,
     validation_data=(X_test, y_test),
     callbacks=[checkpoint],
     verbose=1
    )

# **Saving the model**

In [ ]:
from tensorflow.keras.models import save_model

# Save the model to a file
model.save('artifacts/model.h5')

# **Load the Model Model**

In [ ]:
from tensorflow.keras.models import load_model

# Later, if you want to load the saved model:
loaded_model = load_model('artifacts/model.h5')

# **Saving the Weights**

In [ ]:
model.save_weights('model.h5')

# **Loading the weights**

In [ ]:
model.load_weights('model.h5')

# **Visualization of training over epoch**

In [ ]:
train_acc = history.history['acc']
val_acc = history.history['val_acc']
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(len(train_acc))
plt.plot(epochs, train_acc, 'b', label='Training Accuracy')
plt.plot(epochs, val_acc, 'g', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.grid()
plt.legend()
plt.figure()
plt.show()

plt.plot(epochs, train_loss, 'b', label='Training Loss')
plt.plot(epochs, val_loss, 'g', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.grid()
plt.legend()
plt.show()

# **Confusion Matrix**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# Predict the labels using the trained model
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Convert y_test from one-hot encoding to class indices
y_test_classes = np.argmax(y_test, axis=1)

# Making the Confusion Matrix
cm = confusion_matrix(y_test_classes, y_pred_classes)

target_names = ['cracked-"0"', 'Non Cracked-"1"']

# Plot the confusion matrix as a heatmap with numbers in numeric form
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion matrix')
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
plt.show()

# Calculate and print the classification report
class_names = ['name_a', 'Name_b']  # Replace with your actual class names
report = classification_report(y_test_classes, y_pred_classes, target_names=class_names)
print(report)

# Calculate and print the weighted F1 score
f1 = f1_score(y_test_classes, y_pred_classes, average='weighted')
print(f'Weighted F1 Score: {f1:.4f}')

# **ROC**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate ROC curve and AUC for the positive class (class 1)
fpr, tpr, thresholds = roc_curve(y_test[:, 1], y_pred[:, 1])
roc_auc = roc_auc_score(y_test[:, 1], y_pred[:, 1])

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC curve (area = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], 'k--')  # Diagonal reference line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")  # Add a legend
plt.show()